In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os
sys.path.append('/content/drive/MyDrive/MSCMnet/codes')
print(sorted(os.listdir('/content/drive/MyDrive/MSCMnet/codes')))

Mounted at /content/drive
['__pycache__', 'experiments.py', 'hsi_data.py', 'mscm_modules.py', 'mscm_net.py', 'train.py']


In [ ]:
import os
# List everything in your Google Drive root to find the correct folder name
drive_path = '/content/drive/MyDrive/'
try:
    print(os.listdir(drive_path))
except Exception as e:
    print(f"Error: {e}")

['UHI_Ahmedabad_new', 'Colab Notebooks', 'ISRO Internship Report', 'ISRO_report (3).gdoc', 'ISRO_report (2).gdoc', 'ISRO_report (1).gdoc', 'ISRO_report.gdoc', 'Crochet ', 'Ahmedabad memories ', 'Gossips Girls ', 'Mobile Videos', 'my photos', 'Kokan photos ', 'SAR_Unet', 'Internship_report_bhakti_13_10_2026_final_RAC.gdoc', 'MSCMnet']


In [ ]:
import os
# Inspect the contents of the MSCMnet directory to locate the code files
mscm_path = '/content/drive/MyDrive/MSCMnet'
try:
    print("MSCMnet root contents:", os.listdir(mscm_path))
    if 'codes' in os.listdir(mscm_path):
        print("codes directory contents:", os.listdir(f"{mscm_path}/codes"))
except Exception as e:
    print(f"Error inspecting directory: {e}")

MSCMnet root contents: ['data', 'Notebooks', 'codes', 'checkpoints', 'results']
codes directory contents: ['__pycache__', 'hsi_data.py', 'mscm_modules.py', 'mscm_net.py', 'experiments.py', 'train.py']


In [ ]:
import sys
sys.path.append('/content/drive/MyDrive/MSCMnet/codes')
sys.path_importer_cache.clear()

import importlib, torch
import mscm_modules
importlib.reload(mscm_modules)      # picks up edits to the file
from mscm_modules import Stem, MCSE

x = torch.randn(4, 30, 15, 15)
stem, mcse = Stem(30, 64), MCSE(64, 128)
h = stem(x);  print("stem:", tuple(h.shape))
o = mcse(h);  print("mcse:", tuple(o.shape))
print("params:", sum(p.numel() for m in (stem, mcse) for p in m.parameters()))

stem: (4, 64, 13, 13)
mcse: (4, 128, 13, 13)
params: 62412


In [ ]:
import importlib, torch
sys.path_importer_cache.clear()

import mscm_modules, mscm_net
importlib.reload(mscm_modules); importlib.reload(mscm_net)
from mscm_net import MSCMNet

model = MSCMNet(in_ch=30, num_classes=16)
x = torch.randn(4, 30, 15, 15)
print(model(x).shape)                                   # expect (4, 16)
print(sum(p.numel() for p in model.parameters()))

torch.Size([4, 16])
578140


In [ ]:
import sys, importlib
sys.path_importer_cache.clear()

sys.path.append('/content/drive/MyDrive/MSCMnet/codes')
import hsi_data, mscm_modules, mscm_net, train
for m in (hsi_data, mscm_modules, mscm_net, train):
    importlib.reload(m)

R = '/content/drive/MyDrive/MSCMnet'
D = f'{R}/data/Indian_Pines'
res = train.run(f'{D}/indian_pines_data.mat', f'{D}/indian_pines_gt.mat',
                name='indian_pines', seed=0, epochs=100,
                cache_dir=f'{R}/data/cache', ckpt_dir=f'{R}/checkpoints')

indian_pines seed=0 device=cpu classes=16 train=444 test=9805 params=578,140
ep   1  loss 2.4687  train_acc 25.7%  [59s]


In [ ]:
import os, numpy as np
os.makedirs(f'{R}/results', exist_ok=True)

for seed in range(10):
    out = f'{R}/results/indian_pines_seed{seed}.npz'
    if os.path.exists(out):
        print(f'seed {seed} already done, skipping'); continue
    res = train.run(f'{D}/indian_pines_data.mat', f'{D}/indian_pines_gt.mat',
                    name='indian_pines', seed=seed, epochs=100,
                    cache_dir=f'{R}/data/cache', ckpt_dir=f'{R}/checkpoints')
    np.savez(out, OA=res['OA'], AA=res['AA'], Kappa=res['Kappa'],
             per_class=res['per_class'], cm=res['cm'])

seed 0 already done, skipping
indian_pines seed=1 device=cuda classes=16 train=444 test=9805 params=578,140
ep   1  loss 2.4015  train_acc 30.6%  [2s]
ep  10  loss 0.1067  train_acc 100.0%  [16s]
ep  20  loss 0.0269  train_acc 100.0%  [32s]
ep  30  loss 0.0126  train_acc 100.0%  [48s]
ep  40  loss 0.0081  train_acc 100.0%  [63s]
ep  50  loss 0.0052  train_acc 100.0%  [79s]
ep  60  loss 0.0045  train_acc 100.0%  [94s]
ep  70  loss 0.0027  train_acc 100.0%  [110s]
ep  80  loss 0.0025  train_acc 100.0%  [126s]
ep  90  loss 0.0019  train_acc 100.0%  [141s]
ep 100  loss 0.0017  train_acc 100.0%  [157s]
RESULT indian_pines seed=1: OA 83.40  AA 91.93  Kappa 81.18
indian_pines seed=2 device=cuda classes=16 train=444 test=9805 params=578,140
ep   1  loss 2.4600  train_acc 29.1%  [2s]
ep  10  loss 0.1136  train_acc 100.0%  [16s]
ep  20  loss 0.0260  train_acc 100.0%  [31s]
ep  30  loss 0.0145  train_acc 100.0%  [47s]
ep  40  loss 0.0091  train_acc 100.0%  [62s]
ep  50  loss 0.0060  train_acc 100

In [ ]:
import glob
rs = [np.load(f) for f in sorted(glob.glob(f'{R}/results/indian_pines_seed*.npz'))]
for k in ('OA', 'AA', 'Kappa'):
    v = np.array([float(r[k]) for r in rs])
    print(f'{k:6s} {v.mean():.2f} ± {v.std():.2f}   (min {v.min():.2f}, max {v.max():.2f}, n={len(v)})')

NameError: name 'R' is not defined

In [ ]:
cm = res['cm']          # rows = true class, cols = predicted class
for c in (2, 11):
    row = cm[c-1]
    top = np.argsort(row)[::-1][:4]
    print(f"true class {c}:", {int(t)+1: int(row[t]) for t in top})

true class 2: {2: 1102, 10: 131, 11: 74, 12: 30}
true class 11: {11: 2008, 12: 107, 2: 99, 10: 55}


New updated code traininig

Experiment 1: dropout 0.3

In [ ]:
import sys
sys.path.append('/content/drive/MyDrive/MSCMnet/codes')
sys.path_importer_cache.clear()

import importlib, train, experiments
importlib.reload(train); importlib.reload(experiments)

R = '/content/drive/MyDrive/MSCMnet'
D = f'{R}/data/Indian_Pines'
data, gt = f'{D}/indian_pines_data.mat', f'{D}/indian_pines_gt.mat'

experiments.run_config('ip_drop03', data, gt, R, dropout=0.3)
experiments.compare(R, 'ip_drop03')

ModuleNotFoundError: No module named 'train'